<a href="https://colab.research.google.com/github/roserocarlos/StatAI-Basics/blob/main/Ejercicio4/Cultivating_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cultivating Machine Learning
### Predicción del estado de salud de un cultivo a partir de sensores IoT reales

Dataset: **Plant_health_data.csv** (Kaggle — [gowthamduggirala/plant-health-data](https://www.kaggle.com/datasets/gowthamduggirala/plant-health-data)), 1200 registros reales con 14 columnas de sensores agrícolas.

Objetivo: predecir `Plant_Health_Status` (Healthy / Moderate Stress / High Stress) a partir de 10 sensores numéricos. El notebook está organizado en 8 sesiones incrementales — el código de cada una continúa el de la anterior, y la Sesión 8 usa todo el pipeline acumulado. Al final de cada sesión hay una celda corta de **visualización de apoyo**, separada del código de trabajo, solo para ver de un vistazo lo más representativo de esa sesión.

## Paso 0 — Cargar el dataset

Descarga `Plant_health_data.csv` desde [Kaggle](https://www.kaggle.com/datasets/gowthamduggirala/plant-health-data) a tu computador y súbelo cuando la celda lo pida.

In [ ]:
from google.colab import files
import os

print("Sube tu archivo CSV de Kaggle (Plant_health_data.csv):")
uploaded = files.upload()

os.makedirs('./data', exist_ok=True)
for filename in uploaded.keys():
    os.rename(filename, os.path.join('./data', filename))
    print(f"Archivo '{filename}' guardado en './data/' exitosamente.")

!ls ./data

## Sesión 1 — Del dato al DataFrame

El dato rectangular (Data Frame): fila = registro, columna = feature/predictor. Cargamos el CSV y separamos los 10 sensores numéricos del target categórico `Plant_Health_Status`.

La búsqueda del archivo con `glob` es insensible a mayúsculas/minúsculas: Kaggle a veces exporta el CSV como `plant_health_data.csv` (todo en minúscula) en vez de `Plant_health_data.csv`.

In [ ]:
# ==========================================
# SESION 1 - CARGA Y ESTRUCTURA DE DATOS
# ==========================================
import pandas as pd
import numpy as np
import glob

candidatos = glob.glob("./data/*lant_health_data*.csv") + glob.glob("./data/*Plant_health_data*.csv")
RUTA_CSV = sorted(set(candidatos))[0]
print(f"Archivo detectado: {RUTA_CSV}")

df = pd.read_csv(RUTA_CSV, parse_dates=["Timestamp"])

print("--- SESION 1: Dimensiones del dataset ---")
print(f"Registros (filas): {df.shape[0]} | Caracteristicas (columnas): {df.shape[1]}\n")
print("--- Tipos de datos por sensor ---")
print(df.dtypes)

SENSORES_NUM = ["Soil_Moisture", "Ambient_Temperature", "Soil_Temperature",
                "Humidity", "Light_Intensity", "Soil_pH",
                "Nitrogen_Level", "Phosphorus_Level", "Potassium_Level",
                "Electrochemical_Signal"]
TARGET = "Plant_Health_Status"

**Visualización de apoyo — distribución del target:**

In [ ]:
import matplotlib.pyplot as plt

df[TARGET].value_counts().plot(kind="bar", color="teal", edgecolor="black")
plt.title("Registros por clase de Plant_Health_Status")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()

**Ejercicio:** confirma cuántos registros y columnas tiene tu copia del dataset, y verifica que `SENSORES_NUM` coincide exactamente con los nombres de columnas que ves en `df.dtypes`.

## Sesión 2 — Estadística descriptiva, moda y limpieza según el origen del sensor

Añadimos la **moda** a la media y la mediana. Sobre los nulos: los contamos primero — no se asume que "los sensores IoT siempre fallan"; se revisa. En la descarga real de Kaggle este dataset **no trae valores faltantes**.

Aun así, dejamos una limpieza defensiva por si tu copia del CSV sí trae huecos — y la estrategia depende del **origen físico** de cada sensor, no solo de la estadística: `Soil_Moisture`, `Light_Intensity`, `Ambient_Temperature` y `Humidity` no oscilan alrededor de un centro, siguen un proceso direccional (riego → secado, ciclo diurno). Verificamos que el dataset trae lecturas cada 6 horas sin huecos por planta (`Plant_ID`), así que ahí un valor faltante se interpola dentro de la serie temporal de esa misma planta, en vez de rellenarlo con un promedio global que ignora ese suceso físico. `Soil_pH` y los nutrientes (N, P, K) cambian más lento y sin ese patrón cíclico marcado, así que la mediana global sigue siendo un sustituto razonable para ellos.

In [ ]:
# ==========================================
# SESION 2 - ESTADISTICA, MODA Y LIMPIEZA SEGUN EL ORIGEN DEL SENSOR
# ==========================================
print("--- SESION 2: Estadisticas Descriptivas Iniciales ---")
estadisticas = df[SENSORES_NUM].describe().T[["mean", "50%", "std", "min", "max"]]
estadisticas["moda"] = df[SENSORES_NUM].mode().iloc[0]
print(estadisticas)

nulos_antes = df[SENSORES_NUM].isnull().sum()
print("\nValores nulos detectados por sensor:")
print(nulos_antes)
print(f"\nTotal de nulos en el dataset: {nulos_antes.sum()}")

# La estrategia de limpieza depende del origen fisico de cada sensor, no solo
# de la estadistica. Soil_Moisture, Light_Intensity, Ambient_Temperature y
# Humidity no oscilan alrededor de un centro: siguen un proceso direccional
# (riego/secado, ciclo diurno). Se confirmo que el dataset trae lecturas cada
# 6 horas sin huecos por Plant_ID, asi que ahi un valor faltante se interpola
# dentro de la serie de esa misma planta en vez de usar un promedio global.
# Soil_pH y los nutrientes (N, P, K) cambian mas lento y sin ese patron
# ciclico marcado, asi que la mediana global sigue siendo razonable para ellos.
VARIABLES_TEMPORALES = ["Soil_Moisture", "Light_Intensity", "Ambient_Temperature", "Humidity"]
VARIABLES_LENTAS = ["Soil_Temperature", "Soil_pH", "Nitrogen_Level", "Phosphorus_Level",
                     "Potassium_Level", "Electrochemical_Signal"]

df = df.sort_values(["Plant_ID", "Timestamp"])
for variable in VARIABLES_TEMPORALES:
    df[variable] = df.groupby("Plant_ID")[variable].transform(
        lambda serie: serie.interpolate(method="linear", limit_direction="both")
    )
for variable in VARIABLES_LENTAS:
    df[variable] = df[variable].fillna(df[variable].median())

df = df.dropna(subset=[TARGET])

nulos_despues = df[SENSORES_NUM].isnull().sum()
print("\nValores nulos despues de la limpieza (0 en este dataset: no habia huecos que llenar):")
print(nulos_despues)

**Visualización de apoyo — nulos antes vs. después de imputar:**

In [ ]:
pd.DataFrame({"antes": nulos_antes, "despues": nulos_despues}).plot(
    kind="bar", figsize=(8, 3), color=["#C9552E", "#2E7D32"]
)
plt.title("Valores nulos por sensor: antes vs. despues de la imputacion")
plt.ylabel("Cantidad de nulos")
plt.tight_layout()
plt.show()

**Ejercicio:** compara la media y la mediana de `Light_Intensity`. ¿Cuál sensor tiene mayor diferencia entre ambas? ¿Qué te dice eso sobre la presencia de outliers en ese sensor?

## Sesión 3 — Análisis Exploratorio de Datos (EDA)

Filosofía de Tukey: "el modelo siempre debe seguir a los datos". Calculamos la matriz de correlación de Pearson entre los 10 sensores.

In [ ]:
# ==========================================
# SESION 3 - ANALISIS EXPLORATORIO DE DATOS (EDA)
# ==========================================
print("--- SESION 3: Correlacion entre sensores ---")

matriz_corr = df[SENSORES_NUM].corr()
print("Matriz de correlacion de Pearson (sensores numericos):")
print(matriz_corr.round(2))

**Visualización de apoyo — distribución de Light_Intensity:**

In [ ]:
df["Light_Intensity"].hist(bins=25, color="teal", edgecolor="black", figsize=(6, 3))
plt.title("Distribucion de Intensidad Luminica (lux)")
plt.xlabel("Light_Intensity"); plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

**Ejercicio:** identifica el par de sensores con la correlación más alta (en valor absoluto, sin contar la diagonal) y el par con la correlación más cercana a cero. Interpreta ambos casos en el contexto agronómico del cultivo.

## Sesión 4 — Relaciones y transformación de escala de potencias (Tukey)

`Light_Intensity` tiene cola larga → aplicamos `log10(x + 1)` para linealizar su relación con `Chlorophyll_Content` y comparamos la correlación antes/después.

In [ ]:
# ==========================================
# SESION 4 - TRANSFORMACION Y LINEALIZACION (Tukey)
# ==========================================
print("--- SESION 4: Escala de Potencias sobre Light_Intensity ---")

df["Light_Intensity_log"] = np.log10(df["Light_Intensity"] + 1)

corr_original = df["Light_Intensity"].corr(df["Chlorophyll_Content"])
corr_transf = df["Light_Intensity_log"].corr(df["Chlorophyll_Content"])
print(f"Correlacion original Light vs Clorofila: {corr_original:.4f}")
print(f"Correlacion log(Light) vs Clorofila:      {corr_transf:.4f}")

**Visualización de apoyo — Soil_Moisture vs. Chlorophyll_Content:**

In [ ]:
plt.figure(figsize=(6, 3))
plt.scatter(df["Soil_Moisture"], df["Chlorophyll_Content"], alpha=0.5, color="tomato")
plt.title("Humedad de Suelo vs Contenido de Clorofila")
plt.xlabel("Soil_Moisture (%)"); plt.ylabel("Chlorophyll_Content")
plt.tight_layout()
plt.show()

**Ejercicio:** ¿por qué se usa `log10(x + 1)` y no `log10(x)` directamente? Prueba aplicar la transformación a otro sensor con distribución sesgada (por ejemplo `Potassium_Level`) y observa si mejora alguna correlación de interés.

## Sesión 5 — Partición y estandarización (Z-score)

Estandarizamos con `StandardScaler` ajustado **solo** en entrenamiento (evita fuga de datos) y particionamos 80/20 de forma estratificada.

In [ ]:
# ==========================================
# SESION 5 - PARTICION Y ESTANDARIZACION
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

print("--- SESION 5: Particion de Datos y Escalado Z-Score ---")

predictores = ["Soil_Moisture", "Ambient_Temperature", "Soil_Temperature",
               "Humidity", "Light_Intensity_log", "Soil_pH",
               "Nitrogen_Level", "Phosphorus_Level", "Potassium_Level",
               "Electrochemical_Signal"]

X = df[predictores]
le = LabelEncoder()
y = le.fit_transform(df[TARGET])   # Healthy / Moderate Stress / High Stress -> 0,1,2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Entrenamiento: {X_train_scaled.shape} | Prueba: {X_test_scaled.shape}")
print(f"Clases codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}")

**Visualización de apoyo — tamaño de los conjuntos:**

In [ ]:
plt.bar(["Entrenamiento", "Prueba"], [len(y_train), len(y_test)], color=["#2E7D32", "#C9A227"])
plt.title("Registros por conjunto")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()

**Ejercicio:** quita `stratify=y` de `train_test_split`, vuelve a correr la celda y compara cuántos registros de "High Stress" quedan en el set de prueba. ¿Por qué importa esto?

## Sesión 6 — Modelo lineal vs. Árbol (CART)

Regresión Logística (paramétrica, interpretable) vs. Árbol de Decisión (no paramétrico, captura interacciones no lineales). Comparamos con Accuracy y F1-macro.

In [ ]:
# ==========================================
# SESION 6 - REGRESION LOGISTICA VS CART
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

print("--- SESION 6: Modelos de Linea Base (Logistica vs CART) ---")

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
pred_log = log_model.predict(X_test_scaled)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train_scaled, y_train)
pred_tree = tree_model.predict(X_test_scaled)

acc_log = accuracy_score(y_test, pred_log); f1_log = f1_score(y_test, pred_log, average="macro")
acc_tree = accuracy_score(y_test, pred_tree); f1_tree = f1_score(y_test, pred_tree, average="macro")

print(f"Regresion Logistica -> Accuracy: {acc_log:.4f} | F1-macro: {f1_log:.4f}")
print(f"Arbol CART           -> Accuracy: {acc_tree:.4f} | F1-macro: {f1_tree:.4f}")

**Visualización de apoyo — F1-macro por modelo:**

In [ ]:
plt.bar(["Regresion Logistica", "Arbol CART"], [f1_log, f1_tree], color=["#C9552E", "#2E7D32"])
plt.title("F1-macro: Logistica vs CART")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Ejercicio:** cambia `max_depth` del árbol a 2 y luego a 10. ¿Qué pasa con el F1-macro en cada caso? Relaciónalo con el concepto de sobreajuste (overfitting).

## Sesión 7 — Ensambles: Random Forest y XGBoost

Bagging (árboles en paralelo sobre muestras bootstrap) vs. Boosting (árboles secuenciales que corrigen el error residual). Validamos con 5-fold cross-validation y extraemos la importancia de variables.

In [ ]:
# ==========================================
# SESION 7 - ENSAMBLES (RANDOM FOREST Y XGBOOST)
# ==========================================
!pip install -q xgboost
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

print("--- SESION 7: Ensambles e Importancia de Variables ---")

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_scaled, y_train)
pred_rf = rf_model.predict(X_test_scaled)

xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                           random_state=42, eval_metric="mlogloss")
xgb_model.fit(X_train_scaled, y_train)
pred_xgb = xgb_model.predict(X_test_scaled)

cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="f1_macro")
print(f"Random Forest (F1-macro promedio, 5-Fold CV): {cv_scores.mean():.4f}")

importancias = rf_model.feature_importances_
ranking = sorted(zip(predictores, importancias), key=lambda x: x[1], reverse=True)
print("\nRanking de importancia de variables (Random Forest):")
for var, imp in ranking:
    print(f" -> {var:<24}: {imp*100:5.2f}%")

**Visualización de apoyo — importancia de variables:**

In [ ]:
nombres, valores = zip(*ranking)
plt.barh(nombres[::-1], valores[::-1], color="#2E7D32")
plt.title("Importancia de variables (Random Forest)")
plt.tight_layout()
plt.show()

**Ejercicio:** anota los 3 sensores con mayor importancia en tu corrida. ¿Coinciden con `Soil_pH` y `Soil_Moisture`? Propón una explicación agronómica de por qué esos sensores dominan la predicción.

## Sesión 8 — Pipeline consolidado y evaluación final

Comparamos los 4 modelos, elegimos el mejor por F1-macro y generamos matriz de confusión y `classification_report` del ganador.

In [ ]:
# ==========================================
# SESION 8 - PIPELINE CONSOLIDADO Y EVALUACION FINAL
# ==========================================
from sklearn.metrics import confusion_matrix, classification_report

print("--- SESION 8: Evaluacion y Cierre del Proyecto ---")

acc_rf = accuracy_score(y_test, pred_rf); f1_rf = f1_score(y_test, pred_rf, average="macro")
acc_xgb = accuracy_score(y_test, pred_xgb); f1_xgb = f1_score(y_test, pred_xgb, average="macro")

print("\n=======================================================")
print("        TABLA DE RENDIMIENTO FINAL DE MODELOS          ")
print("=======================================================")
print(f" 1. Regresion Logistica  -> Accuracy: {acc_log:.4f} | F1-macro: {f1_log:.4f}")
print(f" 2. Arbol CART           -> Accuracy: {acc_tree:.4f} | F1-macro: {f1_tree:.4f}")
print(f" 3. Random Forest        -> Accuracy: {acc_rf:.4f} | F1-macro: {f1_rf:.4f}")
print(f" 4. XGBoost              -> Accuracy: {acc_xgb:.4f} | F1-macro: {f1_xgb:.4f}")
print("=======================================================")

resultados = {"Regresion Logistica": f1_log, "Arbol CART": f1_tree,
              "Random Forest": f1_rf, "XGBoost": f1_xgb}
mejor_modelo = max(resultados, key=resultados.get)
print(f"\n[EXITO] Modelo recomendado (mayor F1-macro): {mejor_modelo}")

print("\nMatriz de confusion - mejor modelo (Random Forest):")
print(confusion_matrix(y_test, pred_rf))
print("\nReporte de clasificacion (Random Forest):")
print(classification_report(y_test, pred_rf, target_names=le.classes_))

print("\nEl pipeline incremental de 8 sesiones se ejecuto correctamente de principio a fin.")

**Visualización de apoyo — comparativa final de los 4 modelos:**

In [ ]:
plt.bar(list(resultados.keys()), list(resultados.values()), color=["#C9552E", "#2E7D32", "#7FB88A", "#C9A227"])
plt.title("F1-macro por modelo")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Ejercicio final:** con tus propios resultados, escribe 3-4 líneas de conclusión: ¿qué modelo elegirías para desplegar en campo y por qué, considerando tanto el F1-macro como el recall de la clase "High Stress"?